<a href="https://colab.research.google.com/github/Timoteogon/formacion_de_anillos_en_Marte/blob/main/formacion_de_anillos_en_Marte.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Formación de anillos en Marte

Proyecto personal de mecánica celeste y ciencias planetarias. La pregunta: si la órbita de Fobos sigue contrayéndose, ¿qué le pasaría al llegar al límite de Roche? La hipótesis que quiero explorar es que se fragmentaría y que el material podría formar un anillo alrededor de Marte.

Lo construyo por etapas, validando cada una antes de pasar a la siguiente:

- Etapa I: una órbita de Fobos con el problema de dos cuerpos, integrada numéricamente y comparada con el período real.
- Etapa II: caída hasta el límite de Roche con un modelo de mareas, y tiempo estimado de llegada.
- Etapa III: fragmentación y evolución del anillo (pendiente).

Este notebook cubre las etapas I y II. La animación se hará con la solución analítica $r(t)$ de la etapa II.

## ¿Por qué cae Fobos?

En una aproximación de dos cuerpos, Fobos obedece

$$\ddot{\mathbf r}=-\frac{GM_M}{r^3}\mathbf r,$$

con $G$ la constante de gravitación, $M_M$ la masa de Marte, $\mathbf r$ la posición de Fobos respecto al centro de Marte y $r=|\mathbf r|$. Esta ecuación da una órbita kepleriana: si Fobos parte en una órbita circular y no hay perturbaciones, se queda en ella. La gravedad de un Marte puntual no explica la caída.

Por eso distingo dos descripciones:

- Modelo gravitacional de dos cuerpos: calcula cómo se mueve Fobos bajo la gravedad de Marte (etapa I).
- Modelo de evolución orbital: además incorpora la disipación de las mareas, que es lo que contrae la órbita (etapa II).

Tampoco basta con acercar a Fobos una cantidad fija en cada paso de la animación. Eso fijaría la caída de antemano en vez de calcularla, y la tasa de caída tiene que salir de un modelo físico. Además la caída no es radial: Fobos sigue orbitando mientras su órbita se contrae lentamente, como una sucesión de órbitas casi circulares cada vez más pequeñas.

### Energía y semieje mayor

La energía mecánica específica de una órbita kepleriana es

$$\varepsilon=\frac{v^2}{2}-\frac{GM_M}{r}=-\frac{GM_M}{2a}.$$

Si Fobos pierde energía, $\varepsilon$ se vuelve más negativa y el semieje mayor $a$ disminuye.

Esto no significa que Fobos se frene. Para órbitas circulares de radio $r$:

$$K=\frac{GM_Mm}{2r},\qquad U=-\frac{GM_Mm}{r},\qquad E=K+U=-\frac{GM_Mm}{2r}.$$

Al disminuir $r$, la energía potencial baja el doble de lo que sube la cinética, así que $E$ disminuye aunque la velocidad orbital $v=\sqrt{GM_M/r}$ aumente. Perder energía orbital no equivale a perder velocidad.

### Mecanismo de marea

Fobos levanta una deformación de marea en Marte. Fobos orbita en unas 7.7 h y Marte rota en unas 24.6 h, así que Fobos está por debajo de la órbita sincrónica y se mueve más rápido que la rotación del planeta. Como el interior de Marte disipa energía, el abultamiento de marea se retrasa respecto a la línea Marte–Fobos y queda por detrás del satélite. Su atracción sobre Fobos tiene entonces una componente contraria al movimiento: es un torque negativo que le quita momento angular orbital. Ese momento angular pasa a la rotación de Marte y la energía perdida se disipa como calor en su interior. Con menos momento angular, el semieje mayor disminuye.

## Instalamos dependencias

In [ ]:
!pip install -Uq pymcel

## Librerías a usar

In [ ]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import spiceypy as spy

# Librerias para hacer animaciones
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable



# GM de Fobos en m^3/s^2 (JPL SSD: 0.0007087 +- 0.0000006 km^3/s^2, ref. MAR097)
miu_fobos = 708700

# Periodo orbital real de Fobos en segundos (0.31891023 dias)
periodo_real = 0.31891023 * pc.constantes.dia

# Parámetro gravitacional del sistema Marte-Fobos
mu = pc.constantes.mu_mars + miu_fobos

## Etapa I: una órbita de Fobos

### Posiciones de los cuerpos

Consulto a JPL Horizons los vectores de estado de Marte y de Fobos en J2000 y trabajo con el vector relativo Fobos respecto a Marte.

In [ ]:
try:
    # Intenta hacer la consulta normal a Horizons
    fecha = "2000-01-01 00:00:00"  # fecha estándar en astronomía

    tabla_fobos, jd0, X_fobos_0 = pc.consulta_horizons(
        id="401",
        location="@0",
        epochs=fecha
    )

    tabla_marte, jd0_marte, X_marte_0 = pc.consulta_horizons(
        id="499",
        location="@0",
        epochs=fecha
    )
    print("Consulta a Horizons exitosa.")

except Exception as e:
    print(f"Fallo en la API de la NASA ({e})")
    print("Se usaran valores de respaldo")
    import numpy as np

    # Se usan valores de respaldo
    jd0 = 2451545.0  # Día juliano para el 2000-01-01 00:00:00
    X_fobos_0 = np.array([ 2.06932064e+11, -3.55335086e+09, -5.15007774e+09, -2.36874117e+02,  2.75768632e+04,  1.32647476e+03])
    X_marte_0 = np.array([ 2.06926946e+11, -3.56073080e+09, -5.14791237e+09,  1.30430886e+03,  2.62815889e+04,  5.18846576e+02])

print(f"Día juliano inicial: {jd0}")

### Método de Delambre-Verlet

Resuelvo el problema con el método de diferencias finitas. Aproximo las derivadas por cocientes de diferencias:

$$
\left\{\begin{aligned}
\Delta\vec r_i/\Delta t&\approx\vec v_i\\
\Delta\vec v_i/\Delta t&\approx-\sum_{j\neq i}\frac{\mu_j}{r_{ij}^3}\,\vec r_{ij}
\end{aligned}\right\}_N
$$

con $\Delta\vec r_i=\vec r_i(t_0+\Delta t)-\vec r_i(t_0)$ y $\Delta\vec v_i=\vec v_i(t_0+\Delta t)-\vec v_i(t_0)$. Despejando las posiciones y velocidades desconocidas se obtiene el método de Delambre-Verlet:

$$a(t)=-\sum_{j\neq i}\frac{\mu_j}{r_{ij}(t)^3}\,\vec r_{ij}(t)$$
$$
\left\{\begin{aligned}
\vec r_i(t_0+\Delta t)&\approx\vec r_i(t_0)+\vec v_i(t_0)\Delta t+a(t_0)\Delta t^2/2\\
\vec v_i(t_0+\Delta t)&\approx\vec v_i(t_0)+\left[a(t_0)+a(t_0+\Delta t)\right]\Delta t/2
\end{aligned}\right\}_N
$$

Aquí $N=2$ cuerpos y trabajo con el vector relativo Fobos respecto a Marte, de modo que la suma tiene un solo término con $\mu=GM_M+GM_F$, el parámetro gravitacional del sistema Marte-Fobos. Es un método de segundo orden, así que el error de energía debe escalar como $\Delta t^2$.

In [ ]:
r_fobos_0 = X_fobos_0[:3]
v_fobos_0 = X_fobos_0[3:]
r_marte_0 = X_marte_0[:3]
v_marte_0 = X_marte_0[3:]

def verlet(r, v, deltat, mu):
    # Condiciones iniciales relativas del sistema Marte-Fobos
    r_0 = r
    v_0 = v

    # Aceleracion ahora
    a_0 = -mu * r_0 / np.linalg.norm(r_0)**3

    # Posición después
    r_1 = r_0 + v_0 * deltat + 0.5 * a_0 * deltat**2

    # Aceleración despues

    a_1 = -mu * r_1 / np.linalg.norm(r_1)**3

    # Velocidad después
    v_1 = v_0 + 0.5 * (a_0 + a_1) * deltat
    return r_1, v_1

In [ ]:
a_medio = 9376000 # m, semieje mayor medio de Fobos (JPL SSD)

In [ ]:
T_teorico = 2 * np.pi * np.sqrt(a_medio**3 / mu)

print(f"Período teórico: {T_teorico:.3f} s")
print(f"Período teórico: {T_teorico / 3600:.4f} h")

### Paso temporal

Tomo 365.25 pasos por órbita, igual que para la Tierra (un paso de 1 día en un año de 365.25 días), de modo que

$$\Delta t = \frac{T}{365.25}$$

con $T$ el período teórico calculado arriba. Más adelante compruebo con un estudio de convergencia que este paso es suficiente.

In [ ]:
deltat = T_teorico/365.25
deltat

In [ ]:
N = int(np.ceil(T_teorico / deltat))
tiempos = np.arange(N + 1) * deltat

r_verlet = []
v_verlet = []

# condiciones relativas
r_0 = r_fobos_0 - r_marte_0
v_0 = v_fobos_0 - v_marte_0

r_verlet.append(r_0)
v_verlet.append(v_0)

for i in range(len(tiempos) - 1):
    ri, vi, = verlet(r_0, v_0, deltat, mu)
    r_verlet.append(ri)
    v_verlet.append(vi)
    r_0 = ri
    v_0 = vi

La función `np.ceil()` de NumPy redondea cada elemento de un arreglo hacia arriba (hacia el entero más cercano mayor o igual)[cite: 1].

ejemplo:

```python
import numpy as np

# Arreglo con números decimales (positivos y negativos)[cite: 1]
datos = np.array([1.2, 2.5, 3.8, -1.5])[cite: 1]

# Aplicar np.ceil[cite: 1]
resultado = np.ceil(datos)[cite: 1]

print(resultado)[cite: 1]

[ 2.  3.  4. -1.]

In [ ]:
r_verlet = np.array(r_verlet)
v_verlet = np.array(v_verlet)

In [ ]:
plt.plot(r_verlet[:, 0], r_verlet[:, 1])
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.axis("equal")
plt.grid()
plt.show()

### Conservación de la energía

En el problema de dos cuerpos la energía específica se conserva. Su variación a lo largo de la órbita mide el error del integrador.

In [ ]:
# Energía mecánica específica en cada instante
energia = (np.linalg.norm(v_verlet, axis=1)**2)/2 - mu/np.linalg.norm(r_verlet, axis=1)

# Graficar la energía
plt.plot(tiempos, energia)
plt.xlabel("Tiempo (s)")
plt.ylabel("Energía mecánica específica (J/kg)")
plt.title("Conservación de la energía orbital de Fobos")
plt.grid()
plt.show()

In [ ]:
print(f"Energía inicial: {energia[0]:.12e} J/kg")
print(f"Energía mínima:  {np.min(energia):.12e} J/kg")
print(f"Energía máxima:  {np.max(energia):.12e} J/kg")

In [ ]:
error_relativo = np.max(np.abs(energia - energia[0])) / abs(energia[0])

print(f"Variación relativa máxima: {error_relativo:.3e}")
print(f"Variación porcentual máxima: {error_relativo * 100:.6f} %")

In [ ]:
r_modulo = np.linalg.norm(r_verlet, axis=1)

print(f"Radio inicial: {r_modulo[0]:.3f} m")
print(f"Radio mínimo:  {np.min(r_modulo):.3f} m")
print(f"Radio máximo:  {np.max(r_modulo):.3f} m")
print(f"Radio medio:   {np.mean(r_modulo):.3f} m")

In [ ]:
a_osc = -mu / (2 * energia)

print(f"Semieje mayor inicial: {a_osc[0]:.3f} m")
print(f"Semieje mayor medio:   {np.mean(a_osc):.3f} m")

### Período numérico

Mido el período como el instante en que el ángulo orbital desenrollado completa $2\pi$, interpolando linealmente entre los dos pasos que encierran ese instante. Así la resolución es mejor que $\Delta t$.

In [ ]:
# Ángulo orbital en el plano xy
angulos = np.unwrap(np.arctan2(r_verlet[:, 1], r_verlet[:, 0]))

# Primer instante en que completa una vuelta
objetivo = angulos[0] + 2 * np.pi
indices = np.where(angulos >= objetivo)[0]

if len(indices) > 0:
    i = indices[0]
    paso = (objetivo - angulos[i-1]) / (angulos[i] - angulos[i-1]) * deltat
    T_numerico = tiempos[i-1] + paso

    print(f"Período numérico: {T_numerico:.3f} s")
    print(f"Período numérico: {T_numerico / 3600:.4f} h")
else:
    print("La simulación todavía no completa una vuelta.")

In [ ]:
print(f"Número de estados: {len(r_verlet)}")
print(f"Tiempo integrado: {(len(r_verlet) - 1) * deltat:.3f} s")
print(f"Período teórico: {T_teorico:.3f} s")

### Comparación de períodos

Hay tres períodos distintos y conviene no mezclarlos:

- $T_{\text{teórico}}$: calculado con el semieje mayor medio de JPL.
- $T_{\text{osc}}$: calculado con el semieje mayor osculante de la órbita integrada. Comparar con este mide el error puramente numérico del integrador.
- $T_{\text{real}}$: el período observado de Fobos. Comparar con este mide además lo que el modelo no incluye, sobre todo el achatamiento de Marte ($J_2$).

Por eso se espera que el error respecto al período real sea mayor que el error numérico.

In [ ]:
epsilon = np.abs(T_teorico - T_numerico) / T_teorico * 100

print(f"Epsilon: {epsilon:.3} %")

In [ ]:
T_osc = 2 * np.pi * np.sqrt(a_osc[0]**3 / mu)

epsilon_numerico = np.abs(T_osc - T_numerico) / T_osc * 100
epsilon_real = np.abs(periodo_real - T_numerico) / periodo_real * 100

print(f"Período teórico de la órbita integrada: {T_osc:.3f} s")
print(f"Epsilon numérico: {epsilon_numerico:.3} %")
print(f"Epsilon respecto al período real de Fobos: {epsilon_real:.3} %")

### Método de Kepler

La solución exacta del problema de dos cuerpos en el tiempo se obtiene a partir de los elementos orbitales. Aquí no integro: propago los elementos con la ecuación de Kepler y comparo con Verlet.

#### Elementos orbitales y movimiento medio

Parto del estado inicial relativo $(\vec r_0,\vec v_0)$ de Fobos respecto a Marte y lo convierto en los elementos $a$, $e$, $I$, $\Omega$, $\omega$ y la anomalía verdadera inicial $f_0$. Con ellos obtengo la anomalía media inicial:

$$E_0=2\arctan\left(\sqrt{\frac{1-e}{1+e}}\tan\frac{f_0}{2}\right),\qquad M_0=E_0-e\sin E_0$$

y el movimiento medio, con el mismo $\mu$ del sistema Marte-Fobos que usé en Verlet:

$$n=\sqrt{\frac{\mu}{a^3}}$$

#### Predicción de la posición

Para cada tiempo $t=t_0+\Delta t$ se siguen estos pasos:

- Anomalía media: $M=M_0+n\,\Delta t$, módulo $2\pi$.
- Ecuación de Kepler, resuelta con el método de Newton: $M=E-e\sin E$.
- Anomalía verdadera: $f=2\arctan\left(\sqrt{\frac{1+e}{1-e}}\tan\frac{E}{2}\right)$.
- Posición en el sistema perifocal:

$$r=a\,(1-e\cos E),\qquad x_q=r\cos f,\qquad y_q=r\sin f,\qquad z_q=0$$

- Paso al sistema astronómico, el mismo en que están los vectores de Horizons, con la matriz de rotación $\mathbf R_z(-\Omega)\,\mathbf R_x(-I)\,\mathbf R_z(-\omega)$:

$$\vec r_{\text{ast}}=\mathbf R_z(-\Omega)\,\mathbf R_x(-I)\,\mathbf R_z(-\omega)\,\vec r_{\text{peri}}$$

Es la solución exacta del problema de dos cuerpos para cualquier $t$, sin error de paso temporal. La uso como referencia para Verlet, evaluándola en los mismos tiempos de la integración.

In [ ]:
estado_0 = np.concatenate((r_verlet[0], v_verlet[0]))

p, e, I, W, w, f0 = pc.estado_a_elementos(mu, estado_0)
E0 = 2 * np.arctan(np.sqrt((1 - e) / (1 + e)) * np.tan(f0 / 2))

a_kepler = p / (1 - e**2)
n_kepler = np.sqrt(mu / a_kepler**3)

$E - e\sin E = M$

In [ ]:
E_kepler = []

M0 = E0 - e*np.sin(E0)

# M no empieza en el periapsis
M_kepler = np.mod(M0 + n_kepler*tiempos, 2*np.pi)

for M1 in M_kepler:
    Ei, error, niter = pc.kepler_newton(M1, e, delta=1e-9)
    E_kepler.append(Ei)

E_kepler = np.array(E_kepler)

Con $E$ calculo la distancia al foco, $r = a(1 - e\cos E)$:

In [ ]:
r_kepler = []

for Ei in E_kepler:
    r_kepler.append(a_kepler*(1 - e*np.cos(Ei)))

r_kepler = np.array(r_kepler)

Calculo la anomalia verdadera $f$ a partir de $E$:

$$f = 2\arctan\left(\sqrt{\frac{1+e}{1-e}}\,\tan\frac{E}{2}\right)$$

In [ ]:
f_kepler = []

for Ei in E_kepler:
    fi = 2*np.arctan(np.sqrt((1+e)/(1-e))*np.tan(Ei/2))
    f_kepler.append(fi)

f_kepler = np.array(f_kepler)

In [ ]:
# coordenadas en el sistema perifocal
xq = r_kepler*np.cos(f_kepler)
yq = r_kepler*np.sin(f_kepler)
zq = np.zeros(len(xq))

# matriz de rotacion
rotacion = spy.eul2m(-W, -I, -w, 3, 1, 3)

# Coordenadas en el sistema astronómico
rq_kepler = np.array([xq, yq, zq])
r_kepler_ast = rotacion @ rq_kepler

Graficamos la órbita de Kepler:

In [ ]:
plt.plot(r_kepler_ast[0], r_kepler_ast[1])
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.axis("equal")
plt.grid()
plt.show()

### Comparación Kepler-Verlet

Superpongo las trayectorias de Verlet y de Kepler en el plano $xy$ y calculo el error de posición $|\vec r_{\text{Verlet}}(t)-\vec r_{\text{Kepler}}(t)|$ en cada instante. Como Kepler es la solución exacta del problema de dos cuerpos, esa diferencia mide el error acumulado del integrador. Se espera que crezca con el tiempo, porque Verlet recorre la misma elipse con un pequeño desfase respecto del período exacto.

In [ ]:
r_kepler_ast[:, 0] , r_verlet[0]

In [ ]:
r_kepler_ast[:, -1] , r_verlet[-1]

In [ ]:
diferencia = np.linalg.norm(r_kepler_ast.T - r_verlet, axis=1)

In [ ]:
plt.plot(tiempos, diferencia)
plt.xlim(0, tiempos[-1])
plt.xlabel("Tiempo (s)")
plt.ylabel("Error de posición (m)")
plt.title("Error de posición en el plano xy")
plt.grid()
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=r_kepler_ast[0], y=r_kepler_ast[1],
    mode="lines", name="Kepler",
    line=dict(color="#1f77b4", width=2)
))

fig.add_trace(go.Scatter(
    x=r_verlet[:, 0], y=r_verlet[:, 1],
    mode="lines", name="Verlet",
    line=dict(color="#ff7f0e", width=2, dash="dash"),
    customdata=np.stack([tiempos, diferencia], axis=-1),
    hovertemplate=(
        "x = %{x:.0f} m<br>"
        "y = %{y:.0f} m<br>"
        "t = %{customdata[0]:.0f} s<br>"
        "error = %{customdata[1]:.0f} m"
        "<extra>Verlet</extra>"
    )
))

# Inicio y final de cada órbita
fig.add_trace(go.Scatter(
    x=[r_kepler_ast[0, 0]], y=[r_kepler_ast[1, 0]],
    mode="markers", name="Inicio",
    marker=dict(color="black", size=9, symbol="circle")
))
fig.add_trace(go.Scatter(
    x=[r_kepler_ast[0, -1]], y=[r_kepler_ast[1, -1]],
    mode="markers", name="Final Kepler",
    marker=dict(color="#2ca02c", size=9, symbol="square")
))
fig.add_trace(go.Scatter(
    x=[r_verlet[-1, 0]], y=[r_verlet[-1, 1]],
    mode="markers", name="Final Verlet",
    marker=dict(color="#d62728", size=9, symbol="triangle-up")
))

fig.update_layout(
    template="simple_white",
    title="Órbita de Fobos: Kepler y Verlet",
    xaxis_title="x (m)",
    yaxis_title="y (m)",
    width=700, height=700
)
fig.update_xaxes(showgrid=True, gridcolor="lightgray")
fig.update_yaxes(showgrid=True, gridcolor="lightgray", scaleanchor="x")

fig.show()

#### Nota

Usé Claude (Anthropic) para crear la figura interactiva con plotly.

### Estudio de convergencia

Compruebo que el error de Verlet escala como $\Delta t^2$. Para varios pasos temporales integro una órbita y mido tres errores: el error relativo máximo de la energía, el error del período numérico y el error máximo de posición respecto a la solución exacta de Kepler. En un gráfico log-log, un método de segundo orden da rectas de pendiente cercana a 2.

In [ ]:
def kepler_posiciones(t, r_0, v_0):
    # Elementos orbitales del estado inicial
    estado = np.concatenate((r_0, v_0))
    p, e, I, W, w, f0 = pc.estado_a_elementos(mu, estado)
    E0 = 2 * np.arctan(np.sqrt((1 - e) / (1 + e)) * np.tan(f0 / 2))
    M0 = E0 - e * np.sin(E0)
    a = p / (1 - e**2)
    n = np.sqrt(mu / a**3)

    # Anomalía media y ecuación de Kepler en cada tiempo
    M = np.mod(M0 + n * t, 2 * np.pi)
    E = np.array([pc.kepler_newton(Mi, e, delta=1e-12)[0] for Mi in M])

    # Anomalía verdadera y posición perifocal
    f = 2 * np.arctan(np.sqrt((1 + e) / (1 - e)) * np.tan(E / 2))
    r = a * (1 - e * np.cos(E))
    rq = np.array([r * np.cos(f), r * np.sin(f), np.zeros(len(r))])

    # Paso al sistema astronómico; se devuelve con forma (N, 3), como Verlet
    rotacion = spy.eul2m(-W, -I, -w, 3, 1, 3)
    return (rotacion @ rq).T

In [ ]:
def convergencia(deltat):
    r_0 = r_verlet[0]
    v_0 = v_verlet[0]

    # Período kepleriano de la órbita definida por el estado inicial
    energia_0 = (
        np.linalg.norm(v_0)**2 / 2
        - mu / np.linalg.norm(r_0)
    )
    a_0 = -mu / (2 * energia_0)
    T_0 = 2 * np.pi * np.sqrt(a_0**3 / mu)

    # Se integra 1.1 períodos para garantizar una vuelta completa
    N = int(np.ceil(1.1 * T_0 / deltat))
    tiempos_c = np.arange(N + 1) * deltat

    r_c = np.zeros((N + 1, 3))
    v_c = np.zeros((N + 1, 3))

    r_c[0] = r_0
    v_c[0] = v_0

    # Integración mediante Verlet
    for k in range(N):
        r_c[k + 1], v_c[k + 1] = verlet(
            r_c[k], v_c[k], deltat, mu
        )

    # Error relativo máximo de energía
    energia_c = (
        np.linalg.norm(v_c, axis=1)**2 / 2
        - mu / np.linalg.norm(r_c, axis=1)
    )

    error_energia = (
        np.max(np.abs(energia_c - energia_c[0]))
        / abs(energia_c[0])
    )

    # Base ortonormal del plano orbital
    e_r0 = r_0 / np.linalg.norm(r_0)

    h_vec = np.cross(r_0, v_0)
    h_hat = h_vec / np.linalg.norm(h_vec)

    e_t0 = np.cross(h_hat, e_r0)

    # Coordenadas de la posición en el plano orbital
    coord_x = r_c @ e_r0
    coord_y = r_c @ e_t0

    # Fase orbital continua
    angulos_c = np.unwrap(np.arctan2(coord_y, coord_x))

    # Período numérico por interpolación de fase
    objetivo = angulos_c[0] + 2 * np.pi
    indices = np.where(angulos_c >= objetivo)[0]

    if len(indices) == 0:
        raise ValueError(
            "La integración no completó una vuelta orbital."
        )

    i = indices[0]

    paso = (
        (objetivo - angulos_c[i - 1])
        / (angulos_c[i] - angulos_c[i - 1])
        * deltat
    )

    T_num = tiempos_c[i - 1] + paso
    error_T = abs(T_num - T_0) / T_0

    # Error máximo de posición respecto a la solución de Kepler
    r_kepler_c = kepler_posiciones(tiempos_c, r_0, v_0)

    error_posicion = np.max(
        np.linalg.norm(r_c - r_kepler_c, axis=1)
    )

    return error_energia, T_num, error_T, error_posicion

La función `np.unwrap()` de NumPy se utiliza para "desenrollar" secuencias de ángulos en radianes, corrigiendo saltos abruptos o discontinuidades (mayores a un período, que por defecto es $2\pi$) mediante la suma o resta de múltiplos del período para hacer la curva continua.

ejemplo:

```python
import numpy as np

# Fase con un salto abrupto al cruzar el límite
fase = np.array([1.0, 2.0, 3.0, -3.0, -2.0])

# Aplicar np.unwrap
resultado = np.unwrap(fase)

print(resultado)

[1.         2.         3.         3.28318531 4.28318531]

In [ ]:
pasos_por_orbita = np.array([100, 200, 400, 800, 1600, 3200])
deltats = T_osc / pasos_por_orbita

resultados = np.array([convergencia(dt) for dt in deltats])
errores_energia = resultados[:, 0]
errores_T = resultados[:, 2]
errores_posicion = resultados[:, 3]

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
datos = [
    (errores_energia, "Error relativo máximo de energía", "Energía"),
    (errores_T, "Error relativo del período", "Período"),
    (errores_posicion, "Error máximo de posición (m)", "Posición"),
]

for eje, (errores, etiqueta, nombre) in zip(ax, datos):
    pendiente = np.polyfit(np.log(deltats), np.log(errores), 1)[0]
    eje.loglog(deltats, errores, "o-")
    eje.set_xlabel("Δt (s)")
    eje.set_ylabel(etiqueta)
    eje.set_title(f"{nombre}: pendiente {pendiente:.2f}")
    eje.grid(which="both")

plt.tight_layout()
plt.show()

#### Nota

Usé Claude (Anthropic) como apoyo en esta sección: me ayudó con el código de las funciones y de la gráfica, y con el borrador del análisis.

#### Análisis de la convergencia

Los tres errores (energía, período y posición respecto a Kepler) disminuyen siguiendo rectas en el gráfico log-log, con pendientes de 2.01, 2.00 y 2.00. Esto indica que el error global de Verlet escala como $\Delta t^2$, el comportamiento esperado de un método de segundo orden: al reducir el paso a la mitad, cada error baja aproximadamente a una cuarta parte.

La coincidencia entre las tres magnitudes es importante. La energía y el período se calculan solo con la órbita integrada, mientras que el error de posición usa como referencia la solución exacta de Kepler, independiente del integrador. Que las tres den el mismo orden confirma que el método está bien implementado y que la solución de Kepler es una referencia válida.

En todo el rango estudiado ($\Delta t$ entre T/100 y T/3200) las rectas se mantienen, así que no se llega al régimen donde domina el error de redondeo. El paso que uso en el resto del trabajo, 365.25 pasos por órbita, está dentro de este rango, con errores de energía del orden de $10^{-6}$ y de posición de unos kilómetros sobre un radio de 9238 km.

El error de posición es el de mayor interpretación física: crece durante la órbita porque Verlet acumula un pequeño desfase respecto del período exacto. Es el error que más importaría si Verlet se usara para una integración larga.

## Etapa II: caída hasta el límite de Roche

### Límite de Roche

El límite de Roche es la distancia al centro del planeta por debajo de la cual las fuerzas de marea superan la autogravedad de un satélite. Depende de cómo responda el satélite:

- Satélite rígido, con cohesión interna: $d=1.26\,R_M\left(\dfrac{\rho_M}{\rho_F}\right)^{1/3}$
- Satélite fluido, sin resistencia: $d=2.44\,R_M\left(\dfrac{\rho_M}{\rho_F}\right)^{1/3}$

con $R_M$ y $\rho_M$ el radio y la densidad de Marte, y $\rho_F$ la densidad de Fobos. Fobos es un cuerpo con cohesión, así que el límite rígido es el relevante para su fragmentación y el fluido es una cota superior.

### Forma y densidad de Fobos

Uso las dimensiones aproximadas de Fobos, $27\times22\times18$ km, como diámetros del elipsoide de referencia, con semiejes $a=13.5$ km, $b=11$ km y $c=9$ km. La esfera equivalente tiene el mismo volumen:

$$\frac43\pi R_{\mathrm{eq}}^3=\frac43\pi abc\quad\Rightarrow\quad R_{\mathrm{eq}}=\sqrt[3]{abc}$$

In [ ]:
# Semiejes del elipsoide de referencia (m)
semieje_a = 13.5e3
semieje_b = 11e3
semieje_c = 9e3

R_eq = (semieje_a * semieje_b * semieje_c)**(1/3)
print(f"Radio equivalente: {R_eq / 1e3:.3f} km (JPL, radio medio: 11.08 km)")

El volumen de la esfera equivalente es

$$V_{\mathrm{eq}}=\frac43\pi R_{\mathrm{eq}}^3.$$

La masa de Fobos sale de su $GM$ de JPL, $m_F=GM_F/G$, y la densidad es $\rho_F=m_F/V_{\mathrm{eq}}$.

In [ ]:
V_eq = 4/3 * np.pi * R_eq**3

G = pc.constantes.G
masa_fobos = miu_fobos / G                # kg
masa_marte = pc.constantes.mu_mars / G    # kg

densidad_fobos = masa_fobos / V_eq        # kg/m^3

print(f"Masa de Fobos: {masa_fobos:.4e} kg")
print(f"Densidad de Fobos: {densidad_fobos:.0f} kg/m^3 (JPL: 1872 +- 76 kg/m^3)")

In [ ]:
densidad_marte = 3933 # kg/m^3
radio_marte = 3389.5e3 # m, radio medio de Marte

factor = (densidad_marte / densidad_fobos)**(1/3)
d_rigido = 1.26 * radio_marte * factor
d_fluido = 2.44 * radio_marte * factor

print(f"Límite de Roche rígido: {d_rigido / 1e3:.1f} km")
print(f"Límite de Roche fluido: {d_fluido / 1e3:.1f} km")

### ¿Dónde está Fobos respecto a esos límites?

La comparación se hace con el semieje mayor medio de Fobos, tomado del código y no escrito a mano.

In [ ]:
print(f"Semieje mayor de Fobos: {a_medio / 1e3:.0f} km")
print(f"Dentro del límite fluido: {a_medio < d_fluido}")
print(f"Dentro del límite rígido: {a_medio < d_rigido}")

Fobos orbita dentro del límite fluido pero fuera del rígido. No se ha desintegrado porque su estructura ofrece cierta resistencia a las mareas, por cohesión interna y fricción entre sus materiales. Las estrías y zanjas de su superficie se han relacionado con esfuerzos y fracturas, aunque su origen no está completamente establecido. Por eso uso el límite rígido, $d_{\text{rígido}}$, como distancia de llegada.

### Migración orbital por mareas

Como referencia uso el modelo de Kaula (1964) para una órbita de baja excentricidad y $Q$ constante, tal como lo emplean Lainey et al. (2021):

$$\frac{da}{dt}=-3\,\frac{k_2}{Q}\,\frac{m_F}{M_M}\,n\,a\left(\frac{R_M}{a}\right)^5$$

- $a$: semieje mayor de Fobos.
- $m_F$, $M_M$: masas de Fobos y de Marte.
- $R_M$: radio de Marte.
- $n$: movimiento medio de Fobos.
- $k_2$: número de Love de Marte, su respuesta elástica a la marea.
- $Q$: factor de disipación mareal. Un $Q$ menor implica más disipación.

El signo negativo indica que el semieje mayor disminuye. Lainey et al. (2021), con 14 años de datos de Mars Express, adoptan $k_2=0.169$ y obtienen $Q=93.68\pm0.18$. Ese $Q$ es un valor agrupado: absorbe el efecto de los números de Love de orden superior, que no se modelan por separado.

#### De torque a ecuación para el radio

Esta ecuación sale del momento angular. Para una órbita circular de radio $r$ (con excentricidad $e\approx0.015$, la diferencia con el semieje $a$ es pequeña) la velocidad cumple $v=\sqrt{GM_M/r}$, y el momento angular orbital es

$$L=m_Fvr=m_F\sqrt{GM_M\,r}.$$

El torque es $\tau=dL/dt$. Derivando con $m_F$ y $GM_M$ constantes:

$$\tau=\frac{m_F\sqrt{GM_M}}{2\sqrt r}\frac{dr}{dt}\quad\Rightarrow\quad\frac{dr}{dt}=\frac{2\sqrt r}{m_F\sqrt{GM_M}}\,\tau.$$

En el modelo de marea de equilibrio con desfase pequeño, el torque mareal sobre la órbita de Fobos es negativo, porque Fobos orbita más rápido de lo que Marte rota, y su magnitud es

$$\tau=-\frac32\frac{k_2}{Q}\frac{Gm_F^2R_M^5}{r^6}.$$

Sustituyendo y simplificando, y usando $G\,m_F/\sqrt{GM_M}=(m_F/M_M)\sqrt{GM_M}$:

$$\frac{dr}{dt}=-C\,r^{-11/2},\qquad C=3\,\frac{k_2}{Q}\frac{m_F}{M_M}\sqrt{GM_M}\,R_M^5.$$

Es la misma ecuación de arriba, porque $n\,a\,(R_M/a)^5=\sqrt{GM_M}\,R_M^5\,a^{-11/2}$. Considero $C$ constante en esta primera aproximación.

#### Solución y tiempo hasta Roche

Separo variables, $r^{11/2}\,dr=-C\,dt$, e integro desde $r_0$ en $t=0$:

$$\frac{2}{13}\left(r^{13/2}-r_0^{13/2}\right)=-Ct\quad\Rightarrow\quad\boxed{r(t)=\left[r_0^{13/2}-\frac{13}{2}Ct\right]^{2/13}}$$

Imponiendo $r(t_R)=d_{\text{rígido}}$:

$$\boxed{t_R=\frac{2}{13C}\left(r_0^{13/2}-d_{\text{rígido}}^{13/2}\right)}$$

Como el modelo está escrito en el semieje mayor, en esta etapa tomo $r_0=a_{\text{medio}}$, el semieje mayor medio de Fobos, y no la distancia instantánea en J2000. En las caídas con Kepler y con Verlet uso en cambio el semieje osculante del estado inicial, $r_0=a_{\text{osc}}(0)$, para partir de la misma órbita que ya validé.

In [ ]:
k2 = 0.169  # Lainey et al. (2021)
Q = 93.68   # Lainey et al. (2021), +- 0.18, Q agrupado

C = 3 * (k2 / Q) * (masa_fobos / masa_marte) * np.sqrt(pc.constantes.mu_mars) * radio_marte**5
print(f"C = {C:.6e} (SI)")

### Validación de la tasa con Lainey et al. (2021)

Antes de usar $t_R$ compruebo que el modelo reproduce la aceleración de marea que ellos reportan. Su aceleración secular es el término de la longitud media, $\tfrac12\,dn/dt$, y no $dn/dt$ (su ecuación 3). Con $n^2a^3\simeq GM$ y $dn/dt=-\tfrac32\,(n/a)\,da/dt$, mi modelo da

$$\frac12\frac{dn}{dt}=\frac{9\,n^2k_2\,m_F\,R_M^5}{4\,Q\,M_M\,a^5},$$

que es su ecuación 3. Ellos obtienen $(1.257\pm0.003)\times10^{-3}$ grados/año² con el modelo numérico completo y $1.24\times10^{-3}$ con la aproximación analítica.

In [ ]:
da_dt = -C * a_medio**(-11/2)                 # m/s
n = 2 * np.pi / periodo_real                  # rad/s
dn_dt = -1.5 * (n / a_medio) * da_dt          # rad/s^2

a_por_año = pc.constantes.yr                 # s
medio_dn_dt = 0.5 * dn_dt * (180 / np.pi) * a_por_año**2   # grados/año^2

print(f"da/dt = {da_dt * a_por_año * 100:.2f} cm/año")
print(f"(1/2) dn/dt = {medio_dn_dt:.3e} grados/año^2")
print("Lainey et al. (2021): 1.257e-3 grados/año^2")

In [ ]:
r0 = a_medio        # m, semieje mayor medio de Fobos
r_roche = d_rigido  # m

t_R = 2 / (13 * C) * (r0**(13/2) - r_roche**(13/2))

print(f"r0 = {r0:.6e} m")
print(f"r_roche = {r_roche:.6e} m")
print(f"t_R = {t_R:.6e} s")
print(f"t_R = {t_R / pc.constantes.yr / 1e6:.1f} millones de años")

In [ ]:
r0_verlet = a_osc[0]  # m, semieje mayor osculante del estado inicial
r_roche = d_rigido    # m

t_R_verlet = 2 / (13 * C) * (r0_verlet**(13/2) - r_roche**(13/2))

print(f"r0 = {r0_verlet:.6e} m")
print(f"r_roche = {r_roche:.6e} m")
print(f"t_R = {t_R_verlet:.6e} s")
print(f"t_R = {t_R_verlet / pc.constantes.yr / 1e6:.1f} millones de años")

### Caída de Fobos con Kepler

Con las mismas expresiones de $C$ y de $t_R$ de arriba, calculo el tiempo de llegada al límite de Roche rígido partiendo del semieje osculante del estado inicial, $a_{\text{kepler}}$.

In [ ]:
r0_kepler = a_kepler  # m, semieje mayor osculante del estado inicial
r_roche = d_rigido    # m

t_R_kepler = 2 / (13 * C) * (r0_kepler**(13/2) - r_roche**(13/2))

print(f"r0 = {r0_kepler:.6e} m")
print(f"r_roche = {r_roche:.6e} m")
print(f"t_R = {t_R_kepler:.6e} s")
print(f"t_R = {t_R_kepler / pc.constantes.yr / 1e6:.1f} millones de años")

#### Evolución del semieje mayor

Para recrear la caída uso la solución del modelo de mareas de la etapa II. Parte de la ecuación diferencial del semieje mayor,

$$\frac{da}{dt}=-C\,a^{-11/2},\qquad C=3\,\frac{k_2}{Q}\frac{m_F}{M_M}\sqrt{GM_M}\,R_M^5,$$

que ya validé contra la aceleración de marea reportada por Lainey et al. (2021). Separando variables e integrando desde el semieje inicial $a_0$ en $t=0$, el semieje en cualquier instante es

$$a(t)=\left[a_0^{13/2}-\frac{13}{2}\,C\,t\right]^{2/13}.$$

Aquí tomo $a_0=a_{\text{kepler}}$, el semieje osculante del estado inicial de Horizons, para que la caída parta de la misma órbita que usé en Verlet y en Kepler. Por eso el tiempo de llegada a Roche es de 37.6 millones de años, y no los 37.5 que sale con el semieje medio de JPL.

La caída es lenta frente a la órbita: Fobos tarda unas 7.7 h en dar una vuelta y en ese tiempo el semieje cambia una fracción minúscula. Por eso cada instante puede tratarse como una órbita kepleriana con un semieje $a(t)$ que varía despacio (aproximación adiabática).

Hago estas hipótesis:
- La excentricidad se mantiene constante, aunque las mareas la modificarían.
- Los ángulos $I$, $\Omega$ y $\omega$ no cambian. En la realidad el achatamiento de Marte ($J_2$) los haría precesar.
- La tasa $C$ es constante durante toda la caída.

Evalúo $a(t)$ en 500 instantes entre $t=0$ y el tiempo de llegada al límite de Roche rígido, $t_R$.

In [ ]:
tiempos_R_k = np.linspace(0, t_R_kepler, 500)
a_caida_k = (r0_kepler**(13/2) - (13/2)*C*tiempos_R_k)**(2/13)

In [ ]:
plt.plot(tiempos_R_k / pc.constantes.yr / 1e6, a_caida_k / 1e3)
plt.xlabel("Tiempo (millones de años)")
plt.ylabel("Semieje mayor (km)")
plt.axhline(d_rigido / 1e3, color="red", linestyle="--", label="Límite de Roche rígido")
plt.axhline(d_fluido / 1e3, color="green", linestyle="--", label="Límite de Roche fluido")
plt.legend()
plt.title("Evolución del semieje mayor de Fobos")
plt.xlim(0, tiempos_R_k[-1] / pc.constantes.yr / 1e6)
plt.grid()
plt.show()

### Verlet con marea

Para obtener el tiempo de caída con Verlet agrego a la gravedad de Marte una aceleración de marea. Es una aceleración tangencial, opuesta a la velocidad, $\vec a_{\text{marea}}=-\kappa\,\hat v$. De la energía específica $\varepsilon=-\mu/2a$ se tiene $d\varepsilon/dt=\vec a_{\text{marea}}\cdot\vec v$, y por tanto

$$\frac{da}{dt}=\frac{2a^2}{\mu}\,\vec a_{\text{marea}}\cdot\vec v=-\frac{2a^2}{\mu}\,\kappa\,v.$$

Exigiendo que $da/dt=-C\,a^{-11/2}$ del modelo de mareas, y con $v=\sqrt{\mu/r}$ para una órbita casi circular, se obtiene

$$\kappa(r)=\frac{C\sqrt{\mu}}{2\,r^{7}}.$$

**Aceleración de marea amplificada.** A ritmo real, Fobos tarda unos $4\times10^{10}$ órbitas en llegar a Roche, y no se puede integrar. Multiplico entonces $C$ por un factor $F$ para que la caída dure unas 1000 órbitas, y después deshago la amplificación escalando el tiempo, $t_R=F\,t_{\text{sim}}$. Esto es válido si el semieje cambia poco en cada órbita, y lo compruebo comparando el resultado para distintos $F$.

**Integrador.** La aceleración de marea depende de $\vec v$, que en Delambre-Verlet no se conoce en el instante nuevo. Uso una velocidad provisional, $\vec v_{\text{prov}}=\vec v+\vec a\,\Delta t$, para evaluar $a(t_0+\Delta t)$. El error que introduce es de segundo orden y mucho menor que la propia marea.

**Medida.** Calculo el semieje osculante $a=-\mu/(2\varepsilon)$ una vez por órbita y detengo la simulación cuando cruza $d_{\text{rígido}}$.

In [ ]:
def aceleracion(r, v, mu, C):
    # Gravedad de Marte
    a_grav = -mu * r / np.linalg.norm(r)**3

    # Marea: aceleración tangencial opuesta a la velocidad
    kappa = C * np.sqrt(mu) / (2 * np.linalg.norm(r)**7)
    a_marea = -kappa * v / np.linalg.norm(v)

    return a_grav + a_marea

def verlet_marea(r, v, deltat, mu, C):
    a_0 = aceleracion(r, v, mu, C)

    # Posición después
    r_1 = r + v * deltat + 0.5 * a_0 * deltat**2

    # La aceleración de marea depende de v: se estima con una velocidad provisional
    v_prov = v + a_0 * deltat
    a_1 = aceleracion(r_1, v_prov, mu, C)

    # Velocidad después
    v_1 = v + 0.5 * (a_0 + a_1) * deltat
    return r_1, v_1

In [ ]:
n_orbitas = 1000   # órbitas que dura la caída simulada
F = t_R_verlet / (n_orbitas * T_osc)   # factor de amplificación de la marea
C_amp = F * C
print(f"Factor de amplificación F = {F:.3e}")

r_marea = r_verlet[0].copy()
v_marea = v_verlet[0].copy()

t_sim = [0.0]
a_marea = [a_osc[0]]
t = 0.0
paso = 0
paso_max = int(3 * n_orbitas * 365.25)

while a_marea[-1] > r_roche and paso < paso_max:
    r_marea, v_marea = verlet_marea(r_marea, v_marea, deltat, mu, C_amp)
    t += deltat
    paso += 1

    # Se registra el semieje osculante una vez por órbita
    if paso % 365 == 0:
        energia_m = np.linalg.norm(v_marea)**2 / 2 - mu / np.linalg.norm(r_marea)
        t_sim.append(t)
        a_marea.append(-mu / (2 * energia_m))

t_sim = np.array(t_sim)
a_marea = np.array(a_marea)

# Instante exacto del cruce, por interpolación lineal
t_cruce = t_sim[-2] + (r_roche - a_marea[-2]) / (a_marea[-1] - a_marea[-2]) * (t_sim[-1] - t_sim[-2])

# Se deshace la amplificación para obtener el tiempo físico
t_R_integrado = t_cruce * F

print(f"Pasos de Verlet: {paso}")
print(f"t_R integrado: {t_R_integrado / pc.constantes.yr / 1e6:.2f} millones de años")
print(f"t_R analítico: {t_R_verlet / pc.constantes.yr / 1e6:.2f} millones de años")
print(f"Diferencia: {abs(t_R_integrado - t_R_verlet) / t_R_verlet * 100:.2f} %")

In [ ]:
t_fisico = t_sim * F / pc.constantes.yr / 1e6
a_analitico = (r0_verlet**(13/2) - (13/2) * C * t_sim * F)**(2/13)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(t_fisico[:-1], a_analitico[:-1] / 1e3, "-", label="Analítico")
ax[0].plot(t_fisico[::20], a_marea[::20] / 1e3, "o", label="Verlet con marea")
ax[0].axhline(d_rigido / 1e3, color="red", linestyle="--", label="Límite de Roche rígido")
ax[0].set_xlabel("Tiempo (millones de años)")
ax[0].set_ylabel("Semieje mayor (km)")
ax[0].set_title("Caída de Fobos")
ax[0].legend()
ax[0].grid()

ax[1].plot(t_fisico[:-1], abs(a_marea[:-1] - a_analitico[:-1]) / a_analitico[:-1])
ax[1].set_xlabel("Tiempo (millones de años)")
ax[1].set_ylabel("Diferencia relativa en el semieje")
ax[1].set_title("Verlet con marea frente al analítico")
ax[1].grid()

plt.tight_layout()
plt.show()

#### Análisis de la caída con Verlet y marea

El tiempo de caída hasta el límite de Roche rígido obtenido integrando con Verlet y la aceleración de marea es de 37.57 millones de años, a 0.10 % del valor analítico, 37.61 millones de años. En la figura, el semieje osculante de la simulación sigue la solución analítica durante toda la caída.

La diferencia relativa entre ambos crece con el tiempo hasta cerca de 0.5 %. Se debe a un pequeño desfase temporal de 0.1 %, que al final de la caída, cuando el semieje decrece unas 20 veces más rápido que al inicio, se traduce en una diferencia mayor en el semieje. La aceleración de marea se dedujo para una órbita circular, y Fobos tiene una excentricidad de unos 0.015, lo que es coherente con una diferencia de este orden.

La simulación usa una marea amplificada por un factor $F$ para completar la caída en unas 1000 órbitas, y el tiempo se reescala al final. Esto es válido porque el semieje cambia muy poco en cada órbita, y lo comprobé repitiendo el cálculo con 300 y 3000 órbitas.

#### Nota

Usé Claude (Anthropic) para plantear la aceleración de marea y para el código de esta sección.

### Resumen de los tiempos de caída

Comparo los tres valores de $t_R$ obtenidos: el analítico con el semieje medio de JPL, el analítico con el semieje osculante del estado inicial y el integrado con Verlet y marea.

In [ ]:
valores = {
    "Analítico, semieje medio (JPL)": t_R,
    "Analítico, semieje osculante": t_R_kepler,
    "Verlet con marea": t_R_integrado,
}

for nombre, tiempo in valores.items():
    print(f"{nombre}: {tiempo / pc.constantes.yr / 1e6:.2f} millones de años")

print(f"Diferencia entre semieje medio y osculante: {abs(t_R_kepler - t_R) / t_R * 100:.2f} %")
print(f"Diferencia entre Verlet con marea y analítico (osculante): {abs(t_R_integrado - t_R_kepler) / t_R_kepler * 100:.2f} %")

### Instantáneas de la caída

Para visualizar la caída muestro la órbita de Fobos en distintos instantes entre $t=0$ y $t_R$. Como el semieje cambia muy poco en cada vuelta (aproximación adiabática), cada instantánea es una órbita kepleriana completa con el semieje $a(t_i)$ de ese momento, calculado con la solución de mareas,

$$a(t_i)=\left[a_0^{13/2}-\frac{13}{2}\,C\,t_i\right]^{2/13},$$

y con la misma excentricidad $e$ del estado inicial.

Dibujo cada órbita en el plano orbital, así que no hace falta rotar al sistema astronómico. La ecuación de la elipse con el foco en Marte es

$$r(f)=\frac{a\,(1-e^2)}{1+e\cos f},\qquad x=r\cos f,\quad y=r\sin f,\qquad f\in[0,2\pi].$$

En cada figura aparecen también Marte, como un círculo de radio $R_M$, y los límites de Roche rígido y fluido, como circunferencias de radios $d_{\text{rígido}}$ y $d_{\text{fluido}}$. Muestro una instantánea cada 1 000 000 de años, unas 38 en total, y la última justo en $t_R$. No rotulo cada órbita: el color indica el tiempo, de $t=0$ a $t_R$. Fobos ya está dentro del límite fluido en $t=0$, pero solo se fragmenta al cruzar el rígido, porque es un cuerpo con cohesión.

Esto no es una animación: cada curva es una vuelta completa en un instante distinto de la caída, que dura millones de años. Al inicio las órbitas quedan muy juntas, y al final se separan porque el semieje decrece más rápido. Mantengo la excentricidad y la orientación constantes, como en las hipótesis de la sección anterior.

In [ ]:
orbitas = np.arange(0, t_R_kepler, 1e6 * pc.constantes.yr)
orbitas = np.append(orbitas, t_R_kepler)
f = np.linspace(0, 2 * np.pi, 100)
a_caida_i = (r0_kepler**(13/2) - (13/2)*C*orbitas)**(2/13)

rt = []

for ai in a_caida_i:
    ri = (ai * (1 - e**2) / (1 + e * np.cos(f)))
    rt.append(ri)
rt = np.array(rt)

In [ ]:
fig, ax = plt.subplots()
colores = plt.cm.viridis(np.linspace(0, 1, len(rt)))

for i in range(0, len(rt), 1):
    plt.plot(rt[i] * np.cos(f) / 1e3, rt[i] * np.sin(f) / 1e3, color=colores[i], lw=0.5)

plt.fill(radio_marte * np.cos(f) / 1e3, radio_marte * np.sin(f) / 1e3, color="firebrick", label="Marte")
plt.plot(r_roche * np.cos(f) / 1e3, r_roche * np.sin(f) / 1e3, "k--", lw=1, label="Límite de Roche rígido")
plt.plot(d_fluido * np.cos(f) / 1e3, d_fluido * np.sin(f) / 1e3, "k:", lw=1, label="Límite de Roche fluido")

# Barra de color: tiempo en millones de años
t_final = t_R_kepler / (1e6 * pc.constantes.yr)
sm = ScalarMappable(cmap="viridis", norm=Normalize(vmin=0, vmax=t_final))
fig.colorbar(sm, ax=ax, label="Tiempo (millones de años)")

plt.axis("equal")
plt.xlabel("x (km)")
plt.ylabel("y (km)")
plt.title("Instantáneas de la órbita de Fobos")
plt.grid()
plt.legend(loc="upper right")
plt.show()

En la figura se ve la aceleración de la caída: al inicio las órbitas casi no se distinguen, porque el semieje cambia unos 37 km por cada millón de años, y en los últimos millones de años baja cientos de kilómetros por millón de años, y las curvas se separan.

### Animación de la caída

Para ver la caída como una cámara rápida, animo la órbita de Fobos a lo largo de los 37.6 millones de años. Como una vuelta dura 7.65 horas y la caída millones de años, no es posible mostrar ambas escalas a la vez, así que cada cuadro corresponde a un instante $t_k$ distinto de la caída.

Para que el movimiento se vea parejo, los cuadros no están espaciados uniformemente en el tiempo sino en el semieje, desde $a_0$ hasta $d_{\text{rígido}}$. El tiempo de cada cuadro sale de invertir la solución de mareas,

$$t(a)=\frac{a_0^{13/2}-a^{13/2}}{\tfrac{13}{2}\,C},$$

y la velocidad de la animación queda determinada por el algoritmo: entre dos cuadros pasan $\Delta t_k$ años, y a $f$ cuadros por segundo la velocidad es $\Delta t_k\,f$ millones de años por segundo. Uso $N=300$ cuadros a 30 cuadros por segundo. Al inicio cada cuadro salta unos 0.35 millones de años, unos 10 millones de años por segundo de animación, y al final unos 0.02 millones de años, unos 0.5 millones de años por segundo. El título de cada cuadro muestra el tiempo y la velocidad.

Dibujo la órbita con el semieje $a_k$ de cada cuadro y la excentricidad $e$ constante, como en las instantáneas. Fobos avanza un ángulo fijo de $2\pi/20$ por cuadro. Ese ángulo no es físico: no se puede saber en qué punto de la órbita estará Fobos dentro de millones de años. Lo que sí es real es el semieje $a(t)$ y el tiempo del título. La estela une las posiciones sucesivas de Fobos, por lo que dibuja una espiral y no una órbita.

In [ ]:
N = 300
fps = 30

a_k = np.linspace(r0_kepler, d_rigido, N)
t_k = (r0_kepler**(13/2) - a_k**(13/2)) / ((13/2)*C)

dt_k = np.diff(t_k) / pc.constantes.yr / 1e6
velocidad = dt_k * fps

In [ ]:
plt.rcParams['animation.embed_limit'] = 22   # en MB

# --- Escenario (se dibuja una sola vez, en km) ---
fig, ax = plt.subplots(figsize=(5, 5), dpi=70)
lim = 1.1*r0_kepler/1e3
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')

# Marte y límites fijos
th = np.linspace(0, 2*np.pi, 200)
ax.fill(radio_marte*np.cos(th)/1e3, radio_marte*np.sin(th)/1e3, color='firebrick')
ax.plot(r0_kepler*np.cos(th)/1e3, r0_kepler*np.sin(th)/1e3, 'k--', lw=0.8)
ax.plot(d_rigido*np.cos(th)/1e3, d_rigido*np.sin(th)/1e3, 'r--', lw=0.8)

# Trayectoria completa (calculada una vez)
paso_f = 2*np.pi/20                     # ángulo que avanza Fobos por cuadro (no físico)
sub = 20                                # puntos intermedios por cuadro
kf = np.linspace(0, N-1, (N-1)*sub + 1) # "cuadro continuo"
f_fine = kf * paso_f
a_fine = np.interp(kf, np.arange(N), a_k)
r_fine = a_fine*(1 - e**2)/(1 + e*np.cos(f_fine))
x_fine = r_fine*np.cos(f_fine)/1e3
y_fine = r_fine*np.sin(f_fine)/1e3

# Objetos que se mueven
estela, = ax.plot([], [], '-', alpha=0.3)
fobos,  = ax.plot([], [], 'o')
titulo = ax.set_title('')

# Qué cambia en cada cuadro
def actualizar(k):
    i = k*sub # índice fino que corresponde al cuadro k
    fobos.set_data([x_fine[i]], [y_fine[i]])
    estela.set_data(x_fine[:i+1], y_fine[:i+1])

    t_Myr = t_k[k] / pc.constantes.yr / 1e6
    v = velocidad[max(k-1, 0)]              # velocidad tiene N-1 elementos
    titulo.set_text(f't = {t_Myr:.2f} Myr   |   {v:.2f} Myr/s')
    return estela, fobos, titulo

anim = FuncAnimation(fig, actualizar, frames=N, interval=1000/fps)
plt.close(fig)
HTML(anim.to_jshtml())

La función `np.diff()` de NumPy calcula la diferencia discreta entre elementos consecutivos de un arreglo (es decir, resta cada elemento del siguiente: $x[i+1] - x[i]$).

```python
import numpy as np

a = np.array([1, 2, 4, 7, 0])
resultado = np.diff(a)

print(resultado)

[ 1  2  3 -7]

## Etapa III: fragmentación y anillo.

### Planteamiento

La etapa II termina cuando Fobos llega a $d_{\text{rígido}}$. Aquí estudio qué pasa después, suponiendo que Fobos se ha desintegrado en ese punto: represento el material como una nube de $N=200$ partículas repartidas dentro del elipsoide de Fobos y sigo su evolución bajo la gravedad de Marte. Black y Mittal (2015) predicen que Fobos se desintegrará en un anillo en 20 a 40 millones de años, un rango que contiene el tiempo de caída de 37.6 millones de años que obtuve.

Este primer modelo es el más simple: cada partícula es de prueba, es decir, solo siente a Marte y no se atrae con las demás ni choca con ellas. Así aíslo un solo mecanismo, la rotación diferencial: las partículas más cercanas a Marte orbitan más rápido que las lejanas, y la nube se estira hasta rodear el planeta. No modelo la ruptura misma, que depende de la cohesión y la estructura interna de Fobos; el modelo supone que ya ocurrió.

Para decidir cuándo hay un anillo uso un criterio medible, la continuidad azimutal: el material cubre todos los ángulos alrededor de Marte, sin un sector vacío grande. Otras propiedades de un anillo real, como el aplanamiento, la circularización y el confinamiento radial, necesitan colisiones o lunas pastoras que este modelo no tiene (ver las limitaciones al final de la etapa).

### Nube de partículas

Reparto las partículas con muestreo por rechazo. Sorteo un punto uniforme en la caja que envuelve al elipsoide, $x\in[-a,a]$, $y\in[-b,b]$, $z\in[-c,c]$, y lo acepto si cae dentro,

$$\left(\frac xa\right)^2+\left(\frac yb\right)^2+\left(\frac zc\right)^2\le1.$$

Dividir cada coordenada por su semieje convierte el elipsoide en una esfera de radio 1, y por eso basta comparar la suma de cuadrados con 1. Se acepta cerca del 52 % de los sorteos, el cociente entre los volúmenes del elipsoide y de la caja ($\pi/6$).

Además exijo que cada punto quede a más de $d_{\min}=1.5$ km de los ya aceptados. Sin esa condición, con $N=200$ la distancia mínima entre dos partículas fue de unos 360 m, y si más adelante agrego autogravedad, a esa distancia la atracción sería unas treinta veces mayor que la marea de Marte sobre una separación de 1 km, y el integrador la trataría como un choque. La semilla del generador es fija para que el notebook dé lo mismo cada vez que corre.

In [ ]:
N = 200
d_min = 1.5e3

semiejes = np.array([semieje_a, semieje_b, semieje_c])   # en metros: 13.5e3, 11e3, 9e3
rng = np.random.default_rng(42)        # semilla fija para reproducibilidad


puntos = []
i = 1

while len(puntos) < N:
    p = rng.uniform(-1, 1, 3) * semiejes
    q = p / semiejes
    if q @ q > 1:
        continue
    if len(puntos) > 0:
        d = np.linalg.norm(np.array(puntos) - p, axis=1).min()
        if d <= d_min:
            continue
    puntos.append(p)
puntos = np.array(puntos)


print(len(puntos))
print(puntos[0])
print(np.shape(puntos))

#### Verificación del muestreo

Compruebo que los puntos llenan el elipsoide sin huecos ni bordes rectos con tres proyecciones, que deben ser elipses de semiejes $13.5\times11$, $13.5\times9$ y $11\times9$ km, y que la distancia mínima entre pares es mayor que $d_{\min}$.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))

ax[0].scatter(puntos[:, 0]/1e3, puntos[:, 1]/1e3, s=8)
ax[0].set_xlabel("x (km)")
ax[0].set_ylabel("y (km)")
ax[0].set_title("Puntos aleatorios")
ax[0].set_aspect('equal')
ax[0].grid()

ax[1].scatter(puntos[:, 0]/1e3, puntos[:, 2]/1e3, s=8)
ax[1].set_xlabel("x (km)")
ax[1].set_ylabel("z (km)")
ax[1].set_title("Puntos aleatorios")
ax[1].set_aspect('equal')
ax[1].grid()

ax[2].scatter(puntos[:, 1]/1e3, puntos[:, 2]/1e3, s=8)
ax[2].set_xlabel("y (km)")
ax[2].set_ylabel("z (km)")
ax[2].set_title("Puntos aleatorios")
ax[2].set_aspect('equal')
ax[2].grid()

plt.tight_layout()
plt.show()

In [ ]:
from scipy.spatial.distance import pdist
print(pdist(puntos).min())

### Condiciones iniciales

Pongo a Marte en el origen y el centro de Fobos en $(R,0,0)$, con $R=d_{\text{rígido}}$, en una órbita circular en el plano $xy$. La velocidad circular es $nR$, con $n=\sqrt{\mu/R^3}$, dirigida a lo largo de $y$.

Cada punto $\vec p_i$ del muestreo es la posición relativa al centro de Fobos, de modo que

$$\vec r_i=(R,0,0)+\vec p_i,\qquad \vec v_i=(0,nR,0)+n\,\hat z\times\vec p_i.$$

El último término es la rotación sincrónica: Fobos siempre muestra la misma cara a Marte, así que gira una vez por órbita, con velocidad angular $n$ alrededor de $\hat z$ (perpendicular al plano orbital), y cada partícula lleva la velocidad de esa rotación rígida. El semieje mayor del elipsoide ($x$) queda apuntando a Marte, como en un cuerpo acoplado por marea.

La velocidad de rotación rígida no es la circular a cada distancia, así que cada partícula arranca con un semieje mayor distinto y una excentricidad pequeña. Esa dispersión es la que estira la nube.

A esta distancia el período es $T=2\pi/n=3.39$ h y la velocidad circular es 2.80 km/s. Compruebo que las 200 partículas quedan entre 5433.6 y 5459.1 km de Marte ($R\pm13.5$ km) y que sus velocidades están entre 2.798 y 2.811 km/s.

Simplifico en tres cosas: órbita circular (Fobos tiene $e\approx0.015$), plano orbital sin inclinación, y $\mu=GM_M+GM_F$ de la etapa I, que difiere de $GM_M$ en $2\times10^{-8}$ en términos relativos.

In [ ]:
R = d_rigido
n = np.sqrt(mu / R**3)

v_giro = n * np.cross([0, 0, 1], puntos)


posicion = [R,0,0] + puntos
velocidad = [0,n*R,0] + v_giro


print(R / 1e3, "km")
print(n, "rad/s")
print(2*np.pi / n / 3600, "horas")
print(n * R / 1e3, "km/s")

print(posicion.shape, velocidad.shape)                 # (200, 3) cada uno
print(np.linalg.norm(posicion, axis=1).min()/1e3,
      np.linalg.norm(posicion, axis=1).max()/1e3)      # alrededor de 5446 km, con ±13.5 km
print(np.linalg.norm(velocidad, axis=1).min()/1e3,
      np.linalg.norm(velocidad, axis=1).max()/1e3)     # cerca de 2.80 km/s

### Delambre-Verlet para $N$ partículas

Uso el mismo método de la etapa I, aplicado a las $N$ partículas a la vez. Cada una es un cuerpo de prueba bajo la gravedad de Marte, con $\vec r_i$ medido desde Marte,

$$\vec a_i=-\frac{\mu}{r_i^3}\,\vec r_i.$$

Guardo las posiciones y velocidades en arreglos de forma $(N,3)$ y calculo la aceleración de todas a la vez. La distancia de cada fila sale de `np.linalg.norm(..., axis=1, keepdims=True)`, que da un arreglo $(N,1)$ y hace que cada fila se divida por su propia distancia. El paso es $\Delta t=T/365.25$, con el $T$ de la órbita de $R=d_{\text{rígido}}$.

In [ ]:
r_cuerpos_0 = posicion.copy()
v_cuerpos_0 = velocidad.copy()

def verletCuerpos(r, v, deltat, mu):
    # Condiciones iniciales relativas para los cuerpos del anillo
    r_0 = r
    v_0 = v

    # Aceleracion ahora
    a_0 = -mu * r_0 / np.linalg.norm(r_0, axis=1, keepdims=True)**3

    # Posición después
    r_1 = r_0 + v_0 * deltat + 0.5 * a_0 * deltat**2

    # Aceleración despues

    a_1 = -mu * r_1 / np.linalg.norm(r_1, axis=1, keepdims=True)**3

    # Velocidad después
    v_1 = v_0 + 0.5 * (a_0 + a_1) * deltat
    return r_1, v_1

In [ ]:
print(np.linalg.norm(r_cuerpos_0, axis=1).min()/1e3,
      np.linalg.norm(r_cuerpos_0, axis=1).max()/1e3)

In [ ]:
T = 2*np.pi / n
deltat = T / 365.25
pasos = 366

orbita_anillo = np.linspace(0, 2*np.pi, 100)

r_c = r_cuerpos_0.copy()
v_c = v_cuerpos_0.copy()

energia = []

for oi in range(pasos):
    r_c, v_c = verletCuerpos(r_c, v_c, deltat, mu)
    Ei = 1/2 * np.linalg.norm(v_c, axis=1)**2 - mu / np.linalg.norm(r_c, axis=1)
    energia.append(Ei)

energia = np.array(energia)

In [ ]:
E0 = 1/2 * np.linalg.norm(v_cuerpos_0, axis=1)**2 - mu / np.linalg.norm(r_cuerpos_0, axis=1)

print(f"Error relativo máximo de energía: {np.max(np.abs((energia - E0)/E0)):.2e}")
print(f"Distancia máxima al punto de partida: {np.linalg.norm(r_c - r_cuerpos_0, axis=1).max()/1e3:.0f} km")

#### Verificación del integrador

Como en la etapa I, mido la conservación de la energía específica de cada partícula, $\varepsilon_i=\tfrac12v_i^2-\mu/r_i$, durante una órbita. El error relativo máximo es del orden de $10^{-6}$, parecido al de la etapa I.

Después de 366 pasos, que son 1.002 órbitas, las partículas quedan a unos 530 km del punto de partida. No es un error: unos 70 km vienen de la fracción de órbita de más, y el resto es el comienzo del estiramiento por la rotación diferencial.

### Evolución durante 60 órbitas

Integro 60 órbitas, con 365 pasos de Verlet por órbita, y guardo la posición de las partículas al terminar cada una. Con la dispersión inicial de los semiejes, la rotación diferencial separa a las partículas unos 10° por órbita, así que 60 órbitas bastan para que la nube rodee a Marte.

In [ ]:
n_orbitas = 60
pasos_por_orbita = 365

r_c = r_cuerpos_0.copy()
v_c = v_cuerpos_0.copy()

posiciones = [r_cuerpos_0.copy()]          # órbita 0

for orbita in range(n_orbitas):
    for i in range(pasos_por_orbita):
        r_c, v_c = verletCuerpos(r_c, v_c, deltat, mu)
    posiciones.append(r_c.copy())          # se guarda al terminar cada órbita

posiciones = np.array(posiciones)          # (n_orbitas + 1, 200, 3)

In [ ]:
orbitas_graf = [0, 1, 5, 10]
th = np.linspace(0, 2*np.pi, 200)

fig, ax = plt.subplots(1, 4, figsize=(14, 4))

for a, k in zip(ax, orbitas_graf):
    a.scatter(posiciones[k, :, 0]/1e3, posiciones[k, :, 1]/1e3, s=5)
    a.fill(radio_marte*np.cos(th)/1e3, radio_marte*np.sin(th)/1e3, color="firebrick")
    a.plot(R*np.cos(th)/1e3, R*np.sin(th)/1e3, "k--", lw=0.8)
    a.set_xlabel("x (km)")
    a.set_ylabel("y (km)")
    a.set_title(f"Tras {k} vueltas")
    a.set_aspect("equal")
    a.grid()

plt.tight_layout()
plt.show()

Cada panel muestra la nube justo después de completar un número entero de vueltas, por eso su centro queda siempre cerca del punto de partida. Todas las partículas orbitan a Marte en el mismo sentido, pero con distinta velocidad: las que están más cerca giran más rápido y las más lejanas, más lento. Respecto al centro de la nube, las primeras se adelantan y las segundas se atrasan, y así el arco crece hacia los dos lados. Los dos extremos terminan encontrándose en el lado opuesto de la órbita, y ahí el material rodea a Marte. En la animación de más abajo se ve el movimiento completo, con todas las partículas avanzando en el mismo sentido.

### Hueco angular máximo

Para decidir cuándo el material forma un anillo mido su continuidad azimutal. En cada instante calculo el ángulo de cada partícula alrededor de Marte, $\varphi_i=\arctan2(y_i,x_i)$, los ordeno y miro los huecos entre vecinas consecutivas, incluido el que cierra el círculo (del último ángulo al primero más $2\pi$). El mayor de esos huecos, $\Delta\varphi_{\max}$, vale cerca de 360° si la nube es un punto y baja a pocos grados cuando el material cubre todo el círculo.

Con un número finito de partículas el hueco nunca llega a 0. Para $N$ puntos repartidos al azar sobre un círculo, el hueco máximo esperado es

$$\langle\Delta\varphi_{\max}\rangle\approx\frac{360°}{N}\left(\ln N+\gamma\right)\approx10.6°\quad(N=200),$$

con $\gamma\approx0.577$ la constante de Euler-Mascheroni. Considero el anillo formado cuando $\Delta\varphi_{\max}$ baja de $1.5$ veces ese valor, 15.9°. Un umbral igual al valor esperado lo incumpliría cerca de la mitad de los anillos perfectamente uniformes, y por eso dejo ese margen. Como depende de $N$, hay que recalcularlo si cambio el número de partículas.

In [ ]:
phi = np.arctan2(posiciones[:, :, 1], posiciones[:, :, 0])   # (n_orbitas+1, 200)
s = np.sort(phi, axis=1)

huecos = np.diff(np.append(s, s[:, :1] + 2*np.pi, axis=1), axis=1)
hueco_max = np.degrees(huecos.max(axis=1))                   # (n_orbitas+1,)

In [ ]:
# constante de Euler-Mascheroni
gamma = 0.57721

hueco_esperado = 360/N * (np.log(N) + gamma)    # ≈ 10.6°
umbral = 1.5 * hueco_esperado                    # ≈ 15.9°

nHuecos = np.arange(n_orbitas + 1)

plt.figure(figsize=(10, 5))

plt.plot(nHuecos, hueco_max)
plt.xlabel("Órbitas")
plt.ylabel("Hueco angular máximo (°)")
plt.axhline(hueco_esperado, color="gray", linestyle=":", label="Hueco esperado (uniforme)")
plt.axhline(umbral, color="red", linestyle="--", label="Umbral")
plt.grid()
plt.legend()
plt.show()

In [ ]:
orbita_anillo = np.argmax(hueco_max < umbral)     # primera órbita con hueco < umbral
print(orbita_anillo, "órbitas")
print(orbita_anillo * T / 3600, "horas")
print(orbita_anillo * T / 86400, "días")

#### Resultado del nivel 1: formación del anillo sin autogravedad

Con partículas de prueba, el material de Fobos se estira a lo largo de la órbita por la rotación diferencial: las partículas más cercanas a Marte giran más rápido y las más lejanas más lento. El hueco angular máximo baja casi linealmente, unos 10° por órbita, desde 360° (la nube es un punto) hasta cerca de la órbita 34, cuando el arco rodea a Marte. Después se queda en una meseta de entre 10° y 18°, cerca del hueco esperado de un anillo uniforme de $N=200$ puntos, $\frac{360°}{N}(\ln N+\gamma)\approx10.6°$.

Defino el anillo como formado cuando el hueco máximo baja de $1.5$ veces ese valor, 15.9°, para dejar margen sobre el ruido estadístico de tener pocas partículas. Esto ocurre a las 39 órbitas, unas 132 horas o 5.5 días.

Este resultado depende de la dispersión inicial en el semieje, que viene de imponer una rotación rígida sincrónica a la nube, y no incluye la ruptura misma de Fobos: las partículas no se atraen entre sí, así que el modelo supone que el cuerpo ya está desintegrado. Tampoco hay colisiones, así que no hay aplanamiento ni circularización, y los semiejes y las excentricidades se conservan. Estas limitaciones se discuten al final de la etapa.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# --- Panel 1: vista desde arriba, coloreada por la posición radial inicial ---
sc = ax[0].scatter(posiciones[-1, :, 0]/1e3, posiciones[-1, :, 1]/1e3,
                   c=puntos[:, 0]/1e3, cmap="viridis", s=8)
plt.colorbar(sc, ax=ax[0], label="Posición radial inicial (km)")

ax[0].fill(radio_marte*np.cos(th)/1e3, radio_marte*np.sin(th)/1e3, color="firebrick")
ax[0].plot(R*np.cos(th)/1e3, R*np.sin(th)/1e3, "k--", lw=0.8)
ax[0].set_xlabel("x (km)")
ax[0].set_ylabel("y (km)")
ax[0].set_title(f"Vista desde arriba, órbita {n_orbitas}")
ax[0].set_aspect("equal")
ax[0].grid()

# --- Panel 2: ancho radial del anillo ---
distancia = np.linalg.norm(posiciones[-1], axis=1) / 1e3
phi_final = np.degrees(np.arctan2(posiciones[-1, :, 1], posiciones[-1, :, 0])) % 360
sc2 = ax[1].scatter(phi_final, distancia, c=puntos[:, 0]/1e3, cmap="viridis", s=8)
plt.colorbar(sc2, ax=ax[1], label="Posición radial inicial (km)")
ax[1].set_xlabel("Ángulo (°)")
ax[1].set_ylabel("Distancia a Marte (km)")
ax[1].set_title("Ancho radial del anillo")
ax[1].grid()

plt.tight_layout()
plt.show()

In [ ]:
# Elementos de cada partícula en el instante final (r_c, v_c vienen de la simulación de 60 órbitas)
h = np.cross(r_c, v_c)
e_vec = np.cross(v_c, h) / mu - r_c / np.linalg.norm(r_c, axis=1, keepdims=True)
e_part = np.linalg.norm(e_vec, axis=1)

energia_f = 1/2 * np.linalg.norm(v_c, axis=1)**2 - mu / np.linalg.norm(r_c, axis=1)
a_part = -mu / (2 * energia_f)

print(f"Excentricidad: máxima {e_part.max():.4f}, media {e_part.mean():.4f}")
print(f"Semieje respecto a R: de {(a_part.min() - R)/1e3:.0f} a {(a_part.max() - R)/1e3:.0f} km")
print(f"Distancia a Marte: de {np.linalg.norm(r_c, axis=1).min()/1e3:.0f} a {np.linalg.norm(r_c, axis=1).max()/1e3:.0f} km")

#### Estructura del anillo

A las 60 órbitas el material rodea a Marte (panel izquierdo), en grupos y no de forma uniforme. El color, que indica la posición radial inicial, está mezclado: la rotación diferencial ya enrolló el material más de una vuelta, y al reducir los ángulos a 0–360° las partículas de distintos semiejes se intercalan.

En el panel derecho cada punto es una partícula, con su ángulo y su distancia a Marte. Todas caen sobre una misma curva, como las cuentas de un collar ordenadas por su posición radial inicial, que se enrolla sobre sí misma y por eso sale por un borde y reaparece por el otro. La distancia oscila entre unos 5385 y 5502 km, un ancho de unos 117 km, mayor que los 27 km que tenía la nube al inicio. Las órbitas no son circulares: la rotación rígida inicial le da a cada partícula una excentricidad pequeña, de máximo $\sim0.007$ según la celda anterior, y un semieje distinto, entre $-50$ y $+50$ km respecto a $R$. Como no hay disipación, esos valores no cambian, y un anillo circularizado sería una línea horizontal a $R=5446$ km.

### Animación de la formación del anillo

Muestro las 61 instantáneas, una por órbita completa, como una animación. El título indica la órbita, el tiempo, el hueco máximo y si el material ya cumple el criterio de anillo. Como cada cuadro se toma al completar una vuelta entera, el centro de la nube casi no se mueve de un cuadro a otro (unos 0.25° por vuelta), y lo que se ve es el estiramiento relativo: las partículas internas se adelantan y las externas se atrasan, y el arco crece hacia los dos lados hasta cerrarse en el lado opuesto, hacia la órbita 39.

In [ ]:
fps_anillo = 6                                 # cuadros por segundo (un cuadro por órbita)

# --- Escenario (se dibuja una sola vez, en km) ---
fig, ax = plt.subplots(figsize=(5, 5), dpi=70)
lim = 1.1*R/1e3
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_aspect('equal')
ax.set_xlabel('x (km)')
ax.set_ylabel('y (km)')

th = np.linspace(0, 2*np.pi, 200)
ax.fill(radio_marte*np.cos(th)/1e3, radio_marte*np.sin(th)/1e3, color='firebrick')
ax.plot(R*np.cos(th)/1e3, R*np.sin(th)/1e3, 'k--', lw=0.8)

# --- Objeto que cambia en cada cuadro ---
nube = ax.scatter(posiciones[0, :, 0]/1e3, posiciones[0, :, 1]/1e3,
                  c=puntos[:, 0]/1e3, cmap='viridis', s=8)
titulo = ax.set_title('')

# --- Qué cambia en cada cuadro ---
def actualizar(k):
    nube.set_offsets(posiciones[k, :, :2]/1e3)
    estado = 'anillo formado' if hueco_max[k] < umbral else 'arco'
    titulo.set_text(f'Órbita {k}  |  t = {k*T/86400:.1f} días  |  hueco {hueco_max[k]:.0f}°  ({estado})')
    return nube, titulo

anim = FuncAnimation(fig, actualizar, frames=n_orbitas + 1, interval=1000/fps_anillo)
plt.close(fig)
HTML(anim.to_jshtml())

### Animación del movimiento continuo

Para ver la órbita completa muestro un cuadro cada 15 pasos de Verlet, unos 15° de órbita, durante las 60 órbitas. Se ve que todas las partículas avanzan en el mismo sentido, y que el estiramiento es solo una diferencia de velocidades: las más cercanas a Marte dan la vuelta más rápido.

In [ ]:
plt.rcParams['animation.embed_limit'] = 30  # en MB

pasos_por_cuadro = 15                       # unos 15° de órbita por cuadro
n_cuadros = 60 * 365 // pasos_por_cuadro

r_f, v_f = r_cuerpos_0.copy(), v_cuerpos_0.copy()
posiciones_fina = [r_f.copy()]
for c in range(n_cuadros):
    for i in range(pasos_por_cuadro):
        r_f, v_f = verletCuerpos(r_f, v_f, deltat, mu)
    posiciones_fina.append(r_f.copy())
posiciones_fina = np.array(posiciones_fina)

# --- Animación ---
fig, ax = plt.subplots(figsize=(5, 5), dpi=70)
lim = 1.1*R/1e3
ax.set(xlim=(-lim, lim), ylim=(-lim, lim), aspect='equal', xlabel='x (km)', ylabel='y (km)')

th = np.linspace(0, 2*np.pi, 200)
ax.fill(radio_marte*np.cos(th)/1e3, radio_marte*np.sin(th)/1e3, color='firebrick')
ax.plot(R*np.cos(th)/1e3, R*np.sin(th)/1e3, 'k--', lw=0.8)

nube = ax.scatter(posiciones_fina[0, :, 0]/1e3, posiciones_fina[0, :, 1]/1e3,
                  c=puntos[:, 0]/1e3, cmap='viridis', s=8)

def actualizar(k):
    nube.set_offsets(posiciones_fina[k, :, :2]/1e3)
    ax.set_title(f't = {k*pasos_por_cuadro*deltat/3600:.1f} h')
    return nube,

anim = FuncAnimation(fig, actualizar, frames=n_cuadros + 1, interval=1000/24)
plt.close(fig)
HTML(anim.to_jshtml())

### Limitaciones del modelo

- **Partículas de prueba.** No hay autogravedad ni colisiones. Con ellas el material podría agruparse y el anillo aplanarse y circularizarse. Aquí no hay disipación, así que los semiejes y las excentricidades se conservan, y el ancho radial instantáneo (unos 117 km) lo fijan esas excentricidades iniciales.
- **No modelo la ruptura.** Supongo que Fobos ya está desintegrado al llegar a $d_{\text{rígido}}$ y reparto el material uniformemente dentro de su elipsoide. La resistencia real del cuerpo (cohesión, fricción, estructura interna) no entra.
- **Condición inicial.** La rotación rígida sincrónica fija la dispersión de semiejes (de $-50$ a $+50$ km) y con ella el tiempo en que el anillo se cierra, unos 5.5 días. Otra condición inicial daría otro tiempo.
- **Simplificaciones orbitales.** Órbita circular, sin inclinación y sin $J_2$, y sin marea durante la desintegración, porque la caída dura millones de años y la formación del anillo, días.
- **Criterio de anillo.** El hueco angular mide continuidad y no uniformidad (en la figura se ven grupos), y su umbral depende de $N=200$.
- **Confinamiento radial.** Sin lunas pastoras ni viscosidad, el modelo no tiene un mecanismo que fije el ancho del anillo.

#### Nota

Usé Claude (Anthropic) como asesor en esta etapa: me guió en el planteamiento (muestreo de la nube, condiciones iniciales, criterio del anillo), revisó y corrigió mi código, y me dio el código de las animaciones, de la figura final y de las comprobaciones de energía y excentricidad. El muestreo, el integrador y el cálculo del hueco angular los escribí yo.

### Pendiente

- Nivel 2: autogravedad entre las partículas, con la nube en equilibrio, y comparación con el nivel 1.
- Colisiones y cohesión (nivel 3), que quedan fuera de este trabajo.

## Declaración de uso de IA

Usé asistentes de IA (Claude, de Anthropic, y ChatGPT, de OpenAI) como apoyo para plantear la física del proyecto, revisar el código y los resultados, y redactar las celdas de texto de este notebook. El código de integración lo desarrollé yo. Las constantes provienen de las fuentes citadas abajo.

## Referencias

- Lainey, V. et al. (2021). Mars moon ephemerides after 14 years of Mars Express data. arXiv:2009.06482. https://arxiv.org/abs/2009.06482
- Kaula, W. M. (1964). Tidal dissipation by solid friction and the resulting orbital evolution. Reviews of Geophysics, 2(4), 661-685.
- Black, B. A. y Mittal, T. (2015). The demise of Phobos and development of a Martian ring system. Nature Geoscience, 8(12), 913-917.
- JPL Solar System Dynamics, Planetary Satellite Physical Parameters (radio, densidad y GM de Fobos). https://ssd.jpl.nasa.gov/sats/phys_par/
- JPL Horizons, vectores de estado de Marte y Fobos. https://ssd.jpl.nasa.gov/horizons/